In [2]:
#import packages
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from rdkit import Chem
from rdkit.Chem import Draw

In [3]:
#load, inspect, select and rename data
df_chembl_raw= pd.read_csv("dataCHEMBL.csv", delimiter=";")
df_chembl_raw.head()

#select and rename the columns: 
df_chembl = df_chembl_raw[['Molecule ChEMBL ID', "Molecular Weight", 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'Assay ChEMBL ID']]
df_chembl.columns = ['ChEMBL_ID', 'molecular_weight','Smiles', 'type','relation','value','units','assay_ChEMBL_ID']
df_chembl.head()

#drop all rows which contain no Smiles code
df_chembl.isna().sum() #20 entries in smiles should be deleted
df_chembl= df_chembl.dropna(subset=["Smiles"])
df_chembl.isna().sum()

#select subset of molecular weight between 200-600
df_chembl_subset = df_chembl[(df_chembl["molecular_weight"]>=200) & (df_chembl['molecular_weight']<=400)]

#select a couple of random entries 
df_chembl_subset = df_chembl_subset.sample(n=300, random_state=42)
df_chembl_subset.info()

#store as csv file 
df_chembl_subset.to_csv("dataCHEMBL_cleaned.csv", index=False)

C:\Users\Chiara Hornemann\AppData\Local\Temp\ipykernel_15644\1152899484.py:2: DtypeWarning: Columns (0: Assay Cell Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chembl_raw= pd.read_csv("dataCHEMBL.csv", delimiter=";")


<class 'pandas.DataFrame'>
Index: 300 entries, 64661 to 71001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ChEMBL_ID         300 non-null    str    
 1   molecular_weight  300 non-null    float64
 2   Smiles            300 non-null    str    
 3   type              300 non-null    str    
 4   relation          293 non-null    str    
 5   value             293 non-null    float64
 6   units             293 non-null    str    
 7   assay_ChEMBL_ID   300 non-null    str    
dtypes: float64(2), str(6)
memory usage: 45.1 KB


In [4]:
#combine datasets and inspect 

#load he CHEMBL and the Pubchem data

pubchem_df= pd.read_csv("antibio_pubchem_descr.csv")
chembl_df = pd.read_csv("dataCHEMBL_cleaned.csv")

#pubchem_df.info()
print(pubchem_df.shape)
#chembl_df.info()
print(chembl_df.shape)

# remove duplicates in Smiles in the PubChem data: 
print("Duplicate SMILES in PubChem:",
      pubchem_df["Smiles"].duplicated().sum())

#### modification!! drop duplicates of the same Smiles structure before merging to the ChEMBL dataset. 
pubchem_df = pubchem_df.drop_duplicates(subset=["Smiles"]).copy()

#merge the two datasets
combined_df = pd.concat([pubchem_df, chembl_df],axis=0, ignore_index=True, sort=False)
print ("Shape of combined Dataset:", combined_df.shape)



(298, 9)
(300, 8)
Duplicate SMILES in PubChem: 43
Shape of combined Dataset: (555, 16)


In [5]:
# perform EDA on combined data

# print basic info (shape, first rows, summary statistics)
print("Combined dataset shape:", combined_df.shape)
print("Column names:", combined_df.columns.tolist())
#print("First 5 rows:", combined_df.head(5))
#print("Summary statistics:", combined_df.describe())

# inspect duplicates and missing values 
print("Duplicates:", combined_df.duplicated().sum()) # there are no duplicates 
print("Missing values:", combined_df.isna().sum()) # there are no missing values



Combined dataset shape: (555, 16)
Column names: ['Smiles', 'CID', 'MolecularWeight', 'XLogP', 'TPSA', 'HBondDonorCount', 'HBondAcceptorCount', 'RotatableBondCount', 'HeavyAtomCount', 'ChEMBL_ID', 'molecular_weight', 'type', 'relation', 'value', 'units', 'assay_ChEMBL_ID']
Duplicates: 0
Missing values: Smiles                  0
CID                   300
MolecularWeight       300
XLogP                 316
TPSA                  300
HBondDonorCount       300
HBondAcceptorCount    300
RotatableBondCount    300
HeavyAtomCount        300
ChEMBL_ID             255
molecular_weight      255
type                  255
relation              262
value                 262
units                 262
assay_ChEMBL_ID       255
dtype: int64


In [6]:
# determine most important molecules wit RDKit: 

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

X = combined_df.drop(columns=["value", "ChEMBL_ID", "Smiles"])
y = combined_df["value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("R²:", r2_score(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("train accuracy:", model.score(X_train, y_train))
print("test accuracy:", model.score(X_test, y_test))


ValueError: could not convert string to float: 'MIC'